# Làm sạch dữ liệu Steam games — `games_reduced.csv`

Notebook thực hiện đủ 7 yêu cầu: cleaning log, missing values, duplicates, chuẩn hóa, outliers, dữ liệu sai, validate và so sánh trước-sau.

**Thứ tự chạy** (theo phụ thuộc dữ liệu): Chuẩn hóa → Missing → Duplicates → Dữ liệu sai → Outliers → Validate.

**Nguyên tắc:** không xóa im lặng. Mọi dòng bị xóa và mọi giá trị bị sửa đều có file audit trong `cleaning_output/`. Outliers chỉ được báo cáo, không xóa.

## Cleaning log (vấn đề, nguyên nhân, cách xử lý)

| Vấn đề | Nguyên nhân | Cách xử lý |
|---|---|---|
| **Tags thiếu 33,8%** (42.502 dòng); Developers, Publishers, Genres, Categories thiếu khoảng 7% | Thiếu có cấu trúc: 99,1% game 0 review thiếu Tags, game có review thì 0% thiếu. Khoảng 8.000 game là bản ghi "trống" (mới đăng, ẩn hoặc ngừng bán) | Điền `Unknown` / `Untagged`. Không suy Publisher từ Developer (chỉ giống nhau 71%) |
| **Name thiếu 1 dòng** (id 396420), mọi số liệu bằng 0 | Bản ghi rỗng | Gán `Unknown_396420`, không xóa |
| **Metacritic = 0** (96,6% dòng), **User score = 0** (99,97% dòng) | 0 đang thay cho "chưa có điểm" (điểm thật 1-100); user_score chỉ có 40 giá trị thật | Đổi thành NA. Thêm `positive_ratio` làm thước đo điểm đánh giá thay thế |
| **Owners `0 - 0`** (24.579 dòng) | Nguồn ước lượng không có dữ liệu: cả 24.579 dòng đều 0 review, 0 CCU, 0 playtime; game đã phát hành không thể có 0 owner | Đổi thành NA |
| Owners là chuỗi `20000 - 50000`, ngày là chuỗi `Aug 1, 2023`, tên cột có dấu cách và chữ hoa | Dữ liệu export dạng text | Tách `owners_min/max` + bucket có thứ tự, parse datetime, snake_case, ép kiểu số nhỏ hơn |
| **Text bẩn**: 304 Name có khoảng trắng lạ, 68 Developers thừa dấu phẩy, 96 Categories lặp token | Nhập tay hoặc ghép nhiều nguồn | Chuẩn hóa khoảng trắng, bỏ token rỗng, khử lặp. Không tách Developers theo dấu phẩy (`Accolade, Inc.`) |
| **Duplicates**: không trùng id, không trùng toàn dòng; 8 dòng dư giống hệt trừ `app_id`; 168 dòng cùng tên và ngày nhưng số liệu khác | Dòng giống hệt là listing lặp; nhóm 168 dòng là một sản phẩm bán dưới nhiều app_id | Xóa 8 dòng dư (giữ app_id nhỏ nhất, có audit). Giữ 168 dòng, gắn cờ `multi_listing_flag` |
| **Discount 1-99% trên game giá 0** (35 dòng) | Game chuyển sang free nhưng giá trị giảm giá cũ còn sót | `discount = 0`, ghi audit. 4 dòng 100% giữ nguyên |
| **Owners mâu thuẫn** (13 dòng): Portal 2 có 155.440 review nhưng owners `0 - 20000` | Ước lượng owner gán sai cho app_id khác | Owners thành NA, gắn cờ `owners_invalid` |
| Playtime 2 tuần > tổng (855 dòng); genre Free To Play nhưng price > 0 (139 dòng) | Hai cửa sổ thời gian tính trên tập người chơi khác nhau; hoặc F2P có gói trả phí | Giữ nguyên, chỉ ghi log |
| **Game chưa phát hành** (1 dòng: Roten, 1/12/2026) | Chưa có người chơi hay review thật | **Xóa** theo ngày tham chiếu, có audit |
| **Outliers**: price tối đa 999,98; peak CCU 1.013.936; DLC 3.703; playtime trung bình 3,4 triệu phút | Phân phối lệch nặng; phần lớn là giá trị thật, số ít đáng ngờ (treo máy) | **Không xóa**, chỉ báo cáo bằng log-IQR |
| Đã kiểm tra, không có lỗi | Mojibake: 0 (`IRMÃO` là tiếng Bồ Đào Nha đúng). Ngày mặc định ngày 1: 3,7% so với kỳ vọng 3,3%. `required_age` trong 0-21 | Không cần xử lý |

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# ----- Cấu hình -----
IN_PATH = Path("../data/processed/games_reduced.csv")      # file gốc
OUT_DIR = Path("cleaning_output")        # thư mục kết quả + file audit
REFERENCE_DATE = pd.Timestamp("2026-10-23")  # ngày tham chiếu: game phát hành sau ngày này bị loại
DROP_TECHNICAL_DUPLICATES = True         # xóa dòng giống hệt trừ app_id (False = chỉ gắn cờ)
DROP_UNRELEASED = True                   # xóa game chưa phát hành
MINUTES_PER_YEAR = 525_600
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ---- HELPERS ----
RULE_LOG: list[dict] = []    # nhật ký theo quy tắc (số dòng bị ảnh hưởng, hành động)
CHANGE_LOG: list[dict] = []  # nhật ký từng giá trị bị sửa (audit)


def log_rule(step: str, rule: str, n, action: str) -> None:
    RULE_LOG.append({"step": step, "rule": rule, "n_affected": int(n), "action": action})
    print(f"  [{step}] {rule}: {int(n):,} -> {action}")


def set_values(df: pd.DataFrame, mask: pd.Series, col: str, new, rule: str) -> None:
    """Sửa giá trị có ghi audit (app_id, cột, giá trị cũ, giá trị mới)."""
    mask = mask.fillna(False).astype(bool)
    if not mask.any():
        return
    if new is pd.NA and pd.api.types.is_integer_dtype(df[col]) and not pd.api.types.is_extension_array_dtype(df[col]):
        df[col] = df[col].astype(str(df[col].dtype).capitalize())  # int8 -> Int8 để chứa được NA
    old = df.loc[mask, ["app_id", col]].rename(columns={col: "old_value"})
    old["column"], old["new_value"], old["rule"] = col, str(new), rule
    CHANGE_LOG.extend(old.astype({"old_value": str}).to_dict("records"))
    df.loc[mask, col] = new

## 0. Nạp dữ liệu và profile ban đầu
Trước khi xử lý, kiểm tra ba thứ: số dòng/cột, ô thiếu (NaN) và các số 0 thực chất là "thiếu ẩn".

In [3]:
raw = pd.read_csv(IN_PATH)
print(f"{raw.shape[0]:,} dòng x {raw.shape[1]} cột")

profile = pd.DataFrame({"dtype": raw.dtypes.astype(str), "n_missing": raw.isna().sum(),
                        "pct_missing": (raw.isna().mean() * 100).round(2)})
display(profile[profile["n_missing"] > 0])

hidden = pd.Series({
    "Metacritic score = 0": (raw["Metacritic score"] == 0).sum(),
    "User score = 0": (raw["User score"] == 0).sum(),
    "Estimated owners = '0 - 0'": (raw["Estimated owners"] == "0 - 0").sum(),
}, name="số dòng")
display(hidden.to_frame())
display(raw.head(3))

125,855 dòng x 26 cột


,dtype,n_missing,pct_missing
Name,str,1,0.00
Developers,str,8449,6.71
Publishers,str,8922,7.09
Categories,str,8966,7.12
Genres,str,8423,6.69
Tags,str,42502,33.77


,số dòng
Metacritic score = 0,121597
User score = 0,125815
Estimated owners = '0 - 0',24579


,id,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,Positive,Negative,User score,Metacritic score,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Windows,Mac,Linux
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,True,False,False
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,252,3,0,0,231,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",True,False,False
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,21,3,0,0,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",True,True,False


## 1. Chuẩn hóa tên cột, kiểu dữ liệu, định dạng `[YC4]`
- Tên cột về snake_case (`id` thành `app_id`, `Price` thành `price_usd`, `Average playtime` thành `avg_playtime_*`).
- `Release date` thành datetime; `Estimated owners` thành `owners_bucket` (có thứ tự) cùng `owners_min`, `owners_max` (số).
- Text: chuẩn hóa Unicode, gộp khoảng trắng, bỏ dấu phẩy thừa. Genres, Categories, Tags khử token lặp. Developers và Publishers **không** tách theo dấu phẩy vì tên công ty có dấu phẩy.
- Cột tính thêm: `release_year`, `total_reviews`, `positive_ratio` (điểm đánh giá thay cho `user_score` gần như trống).

In [4]:
_ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\ufeff]")


def to_snake(col: str) -> str:
    name = re.sub(r"[^0-9a-zA-Z]+", "_", col.strip()).strip("_").lower()
    return {"id": "app_id", "price": "price_usd"}.get(name, name.replace("average_", "avg_"))


def clean_text(s: pd.Series) -> pd.Series:
    """NFC, bỏ ký tự zero-width, gộp khoảng trắng, strip; chuỗi rỗng -> NA."""
    s = s.astype("string").str.normalize("NFC")
    s = s.str.replace(_ZERO_WIDTH, "", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()  # \s gồm cả NBSP, tab, dấu cách full-width (U+3000)...
    return s.mask((s == "").fillna(False))


def clean_company(s: pd.Series) -> pd.Series:
    """Developers/Publishers: KHÔNG split theo dấu phẩy vì tên công ty có dấu phẩy ('Accolade, Inc.').
    Chỉ bỏ dấu phẩy thừa ở đầu/cuối và token rỗng (',,')."""
    s = clean_text(s).str.replace(r",(?:\s*,)+", ",", regex=True).str.strip(" ,")
    return s.mask((s == "").fillna(False))


def clean_token_list(s: pd.Series) -> pd.Series:
    """Genres/Categories/Tags: strip từng token, bỏ token rỗng, khử trùng lặp (giữ thứ tự)."""
    def _f(v):
        if pd.isna(v):
            return pd.NA
        toks = dict.fromkeys(t.strip() for t in str(v).split(",") if t.strip())
        return ",".join(toks) if toks else pd.NA
    return clean_text(s).map(_f).astype("string")


def standardize_schema(raw: pd.DataFrame) -> pd.DataFrame:
    print("\n[YC4] Chuẩn hóa schema")
    df = raw.copy()
    df.columns = [to_snake(c) for c in df.columns]

    # --- text ---
    df["name"] = clean_text(df["name"])
    for c in ("developers", "publishers"):
        df[c] = clean_company(df[c])
    for c in ("genres", "categories", "tags"):
        df[c] = clean_token_list(df[c])

    # --- ngày: định dạng gốc 'Aug 1, 2023' ---
    df["release_date"] = pd.to_datetime(df["release_date"], format="%b %d, %Y", errors="coerce")
    log_rule("YC4", "release_date không parse được", df["release_date"].isna().sum(), "để NaT (cần xem lại nếu > 0)")
    df["release_year"] = df["release_date"].dt.year.astype("Int16")

    # --- Estimated owners '20000 - 50000' -> min / max (số) + bucket có thứ tự ---
    parts = df["estimated_owners"].astype("string").str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$")
    df["owners_min"] = pd.to_numeric(parts[0], errors="coerce").astype("Int64")
    df["owners_max"] = pd.to_numeric(parts[1], errors="coerce").astype("Int64")
    log_rule("YC4", "estimated_owners sai định dạng 'min - max'", df["owners_min"].isna().sum(), "để NA")
    order = (df.dropna(subset=["owners_min"]).drop_duplicates("estimated_owners")
               .sort_values("owners_min")["estimated_owners"].tolist())
    df["owners_bucket"] = pd.Categorical(df["estimated_owners"], categories=order, ordered=True)
    df = df.drop(columns="estimated_owners")

    # --- số: ép kiểu + downcast ---
    int_cols = ["app_id", "peak_ccu", "required_age", "discount", "dlc_count", "positive", "negative",
                "user_score", "metacritic_score", "recommendations", "avg_playtime_forever",
                "avg_playtime_two_weeks", "median_playtime_forever", "median_playtime_two_weeks"]
    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="raise", downcast="integer")
    df["price_usd"] = pd.to_numeric(df["price_usd"], errors="raise").round(2)  # đơn vị: USD (đã xác nhận)
    for c in ("windows", "mac", "linux"):
        df[c] = df[c].astype(bool)
    df["total_reviews"] = (df["positive"].astype("int64") + df["negative"].astype("int64")).astype("int32")
    # Tỷ lệ review tích cực = điểm đánh giá thực tế của người chơi (user_score gần như trống). NA khi chưa có review.
    df["positive_ratio"] = (df["positive"] / df["total_reviews"]).where(df["total_reviews"] > 0).round(4)
    print(f"  positive_ratio có giá trị: {df['positive_ratio'].notna().sum():,} dòng "
          f"(user_score thật chỉ có {(df['user_score'] > 0).sum():,} dòng)")

    first = ["app_id", "name", "release_date", "release_year", "owners_bucket", "owners_min", "owners_max",
             "peak_ccu", "required_age", "price_usd", "discount", "dlc_count", "positive", "negative",
             "total_reviews", "positive_ratio", "user_score", "metacritic_score", "recommendations"]
    df = df[first + [c for c in df.columns if c not in first]]
    print(f"  cột sau chuẩn hóa: {list(df.columns)}")
    return df

In [5]:
df = standardize_schema(raw)
display(df.dtypes.astype(str).to_frame("dtype"))
display(df.head(3))


[YC4] Chuẩn hóa schema
  [YC4] release_date không parse được: 0 -> để NaT (cần xem lại nếu > 0)
  [YC4] estimated_owners sai định dạng 'min - max': 0 -> để NA
  positive_ratio có giá trị: 82,956 dòng (user_score thật chỉ có 40 dòng)
  cột sau chuẩn hóa: ['app_id', 'name', 'release_date', 'release_year', 'owners_bucket', 'owners_min', 'owners_max', 'peak_ccu', 'required_age', 'price_usd', 'discount', 'dlc_count', 'positive', 'negative', 'total_reviews', 'positive_ratio', 'user_score', 'metacritic_score', 'recommendations', 'avg_playtime_forever', 'avg_playtime_two_weeks', 'median_playtime_forever', 'median_playtime_two_weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'windows', 'mac', 'linux']


,dtype
app_id,int32
name,string
release_date,datetime64[us]
release_year,Int16
owners_bucket,category
owners_min,Int64
owners_max,Int64
peak_ccu,int32
required_age,int8
price_usd,float64


,app_id,name,release_date,release_year,owners_bucket,owners_min,owners_max,peak_ccu,required_age,price_usd,discount,dlc_count,positive,negative,total_reviews,positive_ratio,user_score,metacritic_score,recommendations,avg_playtime_forever,avg_playtime_two_weeks,median_playtime_forever,median_playtime_two_weeks,developers,publishers,categories,genres,tags,windows,mac,linux
0,2539430,Black Dragon Mage Playtest,2023-08-01,2023,0 - 0,0,0,0,0,0.00,0,0,0,0,0,NaN,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,True,False,False
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,2016,0 - 20000,0,20000,0,0,5.24,65,0,252,3,255,0.9882,0,0,231,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",True,False,False
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,2019,0 - 20000,0,20000,0,0,4.99,0,0,21,3,24,0.8750,0,0,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",True,True,False


## 2. Xử lý missing values theo ý nghĩa nghiệp vụ `[YC2]`
| Cột | Ý nghĩa của giá trị thiếu | Xử lý |
|---|---|---|
| `name` | Bản ghi rỗng | `Unknown_<app_id>`, giữ dòng |
| `developers`, `publishers`, `genres`, `categories` | Chưa có thông tin | `Unknown` (không impute chéo) |
| `tags` | Chưa có review nên chưa có tag | `Untagged` |
| `metacritic_score`, `user_score` | 0 nghĩa là chưa có điểm | NA |
| `owners_*` khi `0 - 0` | Nguồn không có dữ liệu | NA |
| Các cột đếm (`positive`, `peak_ccu`, `playtime`...) | Không phân biệt được "không có" và "bằng 0" | Giữ nguyên số 0 |

In [6]:
def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    print("\n[YC2] Xử lý missing values")
    has_rev = df["total_reviews"] > 0

    # Bằng chứng: Tags thiếu <=> game chưa có review (thiếu có cấu trúc, không phải ngẫu nhiên)
    p0 = df.loc[~has_rev, "tags"].isna().mean()
    p1 = df.loc[has_rev, "tags"].isna().mean()
    print(f"  Bằng chứng: P(Tags thiếu | 0 review) = {p0:.1%} ; P(Tags thiếu | có review) = {p1:.1%}")

    # 1) Name: định danh -> không thể suy ra. Giữ dòng, gán placeholder 'Unknown_<app_id>' (dễ lọc khi phân tích)
    m = df["name"].isna()
    df.loc[m, "name"] = "Unknown_" + df.loc[m, "app_id"].astype("string")
    log_rule("YC2", "name thiếu", m.sum(), "gán 'Unknown_<app_id>' (không xóa dòng)")

    # 2) Developers / Publishers: KHÔNG suy ra publisher = developer (đúng 71% trường hợp != luôn đúng)
    for c in ("developers", "publishers"):
        log_rule("YC2", f"{c} thiếu", df[c].isna().sum(), "điền 'Unknown' (không impute từ cột kia)")
        df[c] = df[c].fillna("Unknown")

    # 3) Genres / Categories: điền 'Unknown'
    for c in ("genres", "categories"):
        log_rule("YC2", f"{c} thiếu", df[c].isna().sum(), "điền 'Unknown'")
        df[c] = df[c].fillna("Unknown")

    # 4) Tags: thiếu vì chưa có review -> 'Untagged'
    log_rule("YC2", "tags thiếu (≈ game chưa có review)", df["tags"].isna().sum(), "điền 'Untagged'")
    df["tags"] = df["tags"].fillna("Untagged")

    # 5) Giá trị 0 thực chất là 'không có dữ liệu' -> NA (0 làm méo mean/median)
    for c in ("metacritic_score", "user_score"):
        z = df[c] == 0
        df[c] = df[c].astype("Int16").mask(z)
        log_rule("YC2", f"{c} = 0 (thang điểm thực 1-100, 0 = chưa có điểm)", z.sum(), "đổi thành NA")

    # 6) owners '0 - 0': đã phát hành thì không thể có 0 owner -> SteamSpy không có dữ liệu
    zero_own = (df["owners_min"] == 0) & (df["owners_max"] == 0)
    activity = (df[["peak_ccu", "total_reviews", "avg_playtime_forever"]] > 0).any(axis=1)
    print(f"  Bằng chứng: dòng owners '0 - 0' nhưng có hoạt động (CCU/review/playtime) = {(zero_own & activity).sum()}")
    df["owners_min"] = df["owners_min"].mask(zero_own)
    df["owners_max"] = df["owners_max"].mask(zero_own)
    df["owners_bucket"] = df["owners_bucket"].mask(zero_own)
    log_rule("YC2", "estimated_owners = '0 - 0'", zero_own.sum(), "đổi owners_min/max/bucket thành NA")

    # 7) Các cột đếm = 0 (positive, negative, peak_ccu, playtime, dlc...) là 0 thật/không hoạt động -> GIỮ 0
    print("  Giữ nguyên số 0 ở các cột đếm (positive, negative, peak_ccu, playtime, dlc_count, recommendations)")
    return df

In [7]:
df = handle_missing(df)
cols = ["name", "developers", "publishers", "genres", "categories", "tags",
        "metacritic_score", "user_score", "owners_min", "owners_max", "owners_bucket"]
display(df[cols].isna().sum().to_frame("số NA sau xử lý"))


[YC2] Xử lý missing values
  Bằng chứng: P(Tags thiếu | 0 review) = 99.1% ; P(Tags thiếu | có review) = 0.0%
  [YC2] name thiếu: 1 -> gán 'Unknown_<app_id>' (không xóa dòng)
  [YC2] developers thiếu: 8,449 -> điền 'Unknown' (không impute từ cột kia)
  [YC2] publishers thiếu: 8,922 -> điền 'Unknown' (không impute từ cột kia)
  [YC2] genres thiếu: 8,423 -> điền 'Unknown'
  [YC2] categories thiếu: 8,966 -> điền 'Unknown'
  [YC2] tags thiếu (≈ game chưa có review): 42,502 -> điền 'Untagged'
  [YC2] metacritic_score = 0 (thang điểm thực 1-100, 0 = chưa có điểm): 121,597 -> đổi thành NA
  [YC2] user_score = 0 (thang điểm thực 1-100, 0 = chưa có điểm): 125,815 -> đổi thành NA
  Bằng chứng: dòng owners '0 - 0' nhưng có hoạt động (CCU/review/playtime) = 0
  [YC2] estimated_owners = '0 - 0': 24,579 -> đổi owners_min/max/bucket thành NA
  Giữ nguyên số 0 ở các cột đếm (positive, negative, peak_ccu, playtime, dlc_count, recommendations)


,số NA sau xử lý
name,0
developers,0
publishers,0
genres,0
categories,0
tags,0
metacritic_score,121597
user_score,125815
owners_min,24579
owners_max,24579


## 3. Đánh giá duplicates trước khi xóa `[YC3]`
Phân loại 5 cấp rồi mới quyết định:
- **A.** trùng toàn bộ dòng; **B.** trùng `app_id`: không có.
- **C.** giống hệt mọi cột trừ `app_id`: listing lặp, **xóa**, giữ `app_id` nhỏ nhất.
- **D.** cùng tên và ngày phát hành nhưng số liệu khác: một sản phẩm nhiều app_id (Call of Duty, Tomb Raider...), **giữ** và gắn cờ `multi_listing_flag`.
- **E.** cùng tên, khác ngày: game khác nhau trùng tên, **giữ**.

In [8]:
def assess_and_remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    print("\n[YC3] Đánh giá duplicates")
    n0 = len(df)
    content_cols = [c for c in df.columns if c != "app_id"]

    # Cấp 1-2: trùng tuyệt đối / trùng khóa
    log_rule("YC3", "A. trùng hoàn toàn mọi cột", df.duplicated().sum(), "không có -> không xử lý")
    log_rule("YC3", "B. trùng app_id", df["app_id"].duplicated().sum(), "không có -> không xử lý")

    # Cấp 3: giống hệt mọi cột TRỪ app_id  => ứng viên 'duplicate kỹ thuật'
    tech_mask = df.duplicated(content_cols, keep=False)
    tech = df[tech_mask].copy()
    tech["dup_group"] = tech.groupby(content_cols, dropna=False, sort=False, observed=True).ngroup()
    tech = tech.sort_values(["dup_group", "app_id"])
    tech["kept_app_id"] = tech.groupby("dup_group")["app_id"].transform("min")
    tech["dup_type"] = "C_technical (giống hệt trừ app_id)"
    tech["action"] = np.where(tech["app_id"] == tech["kept_app_id"], "KEEP (app_id nhỏ nhất)",
                              "DROP" if DROP_TECHNICAL_DUPLICATES else "FLAG only")

    # Cấp 4: cùng name + release_date nhưng nội dung KHÁC => sản phẩm có nhiều app_id (CoD, Tomb Raider...)
    named = ~df["name"].str.startswith("Unknown_")
    same_key = df.duplicated(["name", "release_date"], keep=False) & named
    multi = df[same_key & ~tech_mask].copy()
    multi["dup_group"] = multi.groupby(["name", "release_date"], sort=False).ngroup() + 100_000
    multi["kept_app_id"], multi["dup_type"] = pd.NA, "D_multi_listing (cùng tên+ngày, số liệu khác)"
    multi["action"] = "KEEP (app_id riêng, số liệu khác nhau)"

    # Cấp 5: cùng tên, khác ngày => game trùng tên (homonym) - chỉ thống kê
    homonym = df["name"].duplicated(keep=False) & named & ~same_key
    n_extra = len(tech) - tech["dup_group"].nunique()
    log_rule("YC3", "C. giống hệt trừ app_id (số dòng dư)", n_extra,
             "xóa, giữ app_id nhỏ nhất, ghi audit" if DROP_TECHNICAL_DUPLICATES else "chỉ gắn cờ")
    log_rule("YC3", "D. cùng name+release_date nhưng số liệu khác (số dòng)", len(multi), "GIỮ - gắn cờ multi_listing_flag")
    log_rule("YC3", "E. cùng name khác release_date (số dòng)", homonym.sum(), "GIỮ - là game khác nhau trùng tên")

    cols = ["dup_type", "dup_group", "app_id", "name", "release_date", "price_usd", "positive", "negative",
            "owners_bucket", "developers", "kept_app_id", "action"]
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.concat([tech[cols], multi[cols]]).to_csv(OUT_DIR / "duplicates_review.csv", index=False)

    if DROP_TECHNICAL_DUPLICATES:
        drop_ids = tech.loc[tech["action"] == "DROP", "app_id"]
        df = df[~df["app_id"].isin(drop_ids)].copy()
    df["multi_listing_flag"] = df.duplicated(["name", "release_date"], keep=False) & ~df["name"].str.startswith("Unknown_")
    print(f"  Số dòng: {n0:,} -> {len(df):,} (xóa {n0 - len(df):,})")
    return df

In [9]:
df = assess_and_remove_duplicates(df)
review = pd.read_csv(OUT_DIR / "duplicates_review.csv")
display(review.groupby(["dup_type", "action"]).agg(số_dòng=("app_id", "size"), số_nhóm=("dup_group", "nunique")))
display(review[review["dup_type"].str.startswith("C")][["dup_group", "app_id", "name", "kept_app_id", "action"]])


[YC3] Đánh giá duplicates
  [YC3] A. trùng hoàn toàn mọi cột: 0 -> không có -> không xử lý
  [YC3] B. trùng app_id: 0 -> không có -> không xử lý
  [YC3] C. giống hệt trừ app_id (số dòng dư): 8 -> xóa, giữ app_id nhỏ nhất, ghi audit
  [YC3] D. cùng name+release_date nhưng số liệu khác (số dòng): 168 -> GIỮ - gắn cờ multi_listing_flag
  [YC3] E. cùng name khác release_date (số dòng): 2,039 -> GIỮ - là game khác nhau trùng tên
  Số dòng: 125,855 -> 125,847 (xóa 8)


số_dòng  số_nhóm
dup_type                                      action                                                  
C_technical (giống hệt trừ app_id)            DROP                                          8        5
                                              KEEP (app_id nhỏ nhất)                        5        5
D_multi_listing (cùng tên+ngày, số liệu khác) KEEP (app_id riêng, số liệu khác nhau)      168       66

,dup_group,app_id,name,kept_app_id,action
0,0,37960,Jewel Quest Pack,37960.0,KEEP (app_id nhỏ nhất)
1,0,37970,Jewel Quest Pack,37960.0,DROP
2,0,37980,Jewel Quest Pack,37960.0,DROP
3,0,37990,Jewel Quest Pack,37960.0,DROP
4,1,39360,Freeze Tag Fun Pack #1,39360.0,KEEP (app_id nhỏ nhất)
5,1,39391,Freeze Tag Fun Pack #1,39360.0,DROP
6,2,2186000,Nautical Dispute,2186000.0,KEEP (app_id nhỏ nhất)
7,2,2377910,Nautical Dispute,2186000.0,DROP
8,3,38000,Mahjong Quest Collection,38000.0,KEEP (app_id nhỏ nhất)
9,3,38010,Mahjong Quest Collection,38000.0,DROP


## 4. Xử lý dữ liệu sai hoặc không hợp lệ `[YC6]`
| Rule | Nội dung | Hành động |
|---|---|---|
| R1 | Vi phạm phạm vi (price < 0, discount ngoài 0-100, điểm ngoài 1-100, tuổi ngoài 0-21, số đếm âm) | NA (hiện 0 dòng) |
| R2 | Giá 0 nhưng discount 1-99% | `discount = 0` |
| R3 | `release_date` sau ngày tham chiếu (chưa phát hành) | **Xóa dòng** |
| R4 | Playtime vượt thời gian kể từ khi phát hành, hoặc 2 tuần vượt 14 ngày | NA (hiện 0 dòng) |
| R5 | `total_reviews` hoặc `peak_ccu` lớn hơn `owners_max` | Owners thành NA, cờ `owners_invalid` |
| R6, R7 | Playtime 2 tuần lớn hơn tổng; F2P nhưng có giá | Chỉ ghi log |

In [10]:
def fix_invalid(df: pd.DataFrame) -> pd.DataFrame:
    print("\n[YC6] Xử lý dữ liệu sai / không hợp lệ")

    # R1 phạm vi giá trị (domain)
    n_changes_start = len(CHANGE_LOG)
    set_values(df, (df["price_usd"] < 0), "price_usd", np.nan, "R1 price < 0")
    set_values(df, ~df["discount"].between(0, 100), "discount", pd.NA, "R1 discount ngoài 0-100")
    for c in ("metacritic_score", "user_score"):
        set_values(df, df[c].notna() & ~df[c].between(1, 100), c, pd.NA, f"R1 {c} ngoài 1-100")
    set_values(df, ~df["required_age"].between(0, 21), "required_age", pd.NA, "R1 required_age ngoài 0-21")
    for c in ["peak_ccu", "dlc_count", "positive", "negative", "recommendations"]:
        set_values(df, df[c] < 0, c, pd.NA, f"R1 {c} < 0")
    log_rule("YC6", "R1 vi phạm phạm vi (price/discount/score/age/count)", len(CHANGE_LOG) - n_changes_start,
             "đặt NA (nếu > 0)")

    # R2 game miễn phí không thể giảm giá theo % (1-99%); 100% giữ vì là khuyến mãi tặng free
    r2 = (df["price_usd"] == 0) & df["discount"].between(1, 99)
    set_values(df, r2, "discount", 0, "R2 price=0 nhưng discount 1-99%")
    log_rule("YC6", "R2 price=0 & discount 1-99%", r2.sum(), "discount -> 0 (audit trong value_changes.csv); 100% giữ nguyên")

    # R3 game chưa phát hành (release_date > ngày tham chiếu): chưa có người chơi/review thật -> xóa, có audit
    unrel = df["release_date"] > REFERENCE_DATE
    df.loc[unrel, ["app_id", "name", "release_date", "price_usd"]].to_csv(OUT_DIR / "unreleased_removed.csv", index=False)
    log_rule("YC6", f"R3 release_date > {REFERENCE_DATE.date()} (chưa phát hành)", unrel.sum(),
             "XÓA dòng (audit: unreleased_removed.csv)" if DROP_UNRELEASED else "chỉ thống kê, giữ")
    if DROP_UNRELEASED:
        df = df[~unrel].copy()

    # R4 playtime không thể vượt thời gian từ lúc phát hành tới nay; 2 tuần không thể > 20.160 phút
    age_min = ((REFERENCE_DATE - df["release_date"]).dt.days.clip(lower=0) + 1) * 1440
    n4 = 0
    for c in ("avg_playtime_forever", "median_playtime_forever"):
        bad = df[c] > age_min
        n4 += int(bad.sum()); set_values(df, bad, c, pd.NA, "R4 playtime > thời gian kể từ phát hành")
    for c in ("avg_playtime_two_weeks", "median_playtime_two_weeks"):
        bad = df[c] > 14 * 1440
        n4 += int(bad.sum()); set_values(df, bad, c, pd.NA, "R4 playtime 2 tuần > 14 ngày")
    log_rule("YC6", "R4 playtime bất khả thi về mặt vật lý", n4, "đặt NA (hiện không có dòng nào)")

    # R5 owners mâu thuẫn: số review hoặc CCU đỉnh không thể lớn hơn cận trên số owner
    omax = df["owners_max"].astype("float64")
    r5 = ((df["total_reviews"] > omax) | (df["peak_ccu"] > omax)).fillna(False)
    df["owners_invalid"] = r5
    for c in ("owners_min", "owners_max"):
        set_values(df, r5, c, pd.NA, "R5 reviews/peak_ccu > owners_max")
    df["owners_bucket"] = df["owners_bucket"].mask(r5)
    log_rule("YC6", "R5 total_reviews hoặc peak_ccu > owners_max", r5.sum(), "owners -> NA + cờ owners_invalid (thường do SteamSpy gán owner cho app_id khác)")

    # R6, R7: mâu thuẫn mềm -> chỉ ghi log, KHÔNG sửa và KHÔNG thêm cột
    r6 = ((df["avg_playtime_two_weeks"] > df["avg_playtime_forever"]) |
          (df["median_playtime_two_weeks"] > df["median_playtime_forever"])).fillna(False)
    log_rule("YC6", "R6 playtime 2 tuần > playtime tổng (2 cửa sổ thời gian tính trên tập người chơi khác nhau)", r6.sum(), "GIỮ nguyên, chỉ ghi log")
    r7 = df["genres"].str.contains("Free To Play", regex=False) & (df["price_usd"] > 0)
    log_rule("YC6", "R7 genre 'Free To Play' nhưng price_usd > 0 (có thể F2P có gói trả phí)", r7.sum(), "GIỮ nguyên, chỉ ghi log")
    return df

In [11]:
df = fix_invalid(df)
display(pd.DataFrame(CHANGE_LOG).groupby("rule").size().to_frame("số giá trị bị sửa"))
display(pd.read_csv(OUT_DIR / "unreleased_removed.csv"))


[YC6] Xử lý dữ liệu sai / không hợp lệ
  [YC6] R1 vi phạm phạm vi (price/discount/score/age/count): 0 -> đặt NA (nếu > 0)
  [YC6] R2 price=0 & discount 1-99%: 35 -> discount -> 0 (audit trong value_changes.csv); 100% giữ nguyên
  [YC6] R3 release_date > 2026-10-23 (chưa phát hành): 1 -> XÓA dòng (audit: unreleased_removed.csv)
  [YC6] R4 playtime bất khả thi về mặt vật lý: 0 -> đặt NA (hiện không có dòng nào)
  [YC6] R5 total_reviews hoặc peak_ccu > owners_max: 13 -> owners -> NA + cờ owners_invalid (thường do SteamSpy gán owner cho app_id khác)
  [YC6] R6 playtime 2 tuần > playtime tổng (2 cửa sổ thời gian tính trên tập người chơi khác nhau): 855 -> GIỮ nguyên, chỉ ghi log
  [YC6] R7 genre 'Free To Play' nhưng price_usd > 0 (có thể F2P có gói trả phí): 139 -> GIỮ nguyên, chỉ ghi log


,số giá trị bị sửa
rule,
R2 price=0 nhưng discount 1-99%,35
R5 reviews/peak_ccu > owners_max,26


,app_id,name,release_date,price_usd
0,4418940,Roten,2026-12-01,0.0


## 5. Kiểm tra outliers, không xóa `[YC5]`
Các cột này có 21% đến 97% giá trị bằng 0, nên IQR trên dữ liệu thô vô nghĩa (Q1 = Q3 = 0). Phương pháp: **IQR trên log1p của các giá trị lớn hơn 0**, ngưỡng cực trị là Q3 + 3·IQR. Kết quả chỉ để báo cáo, không xóa và không thêm cột vào dữ liệu.

In [12]:
OUTLIER_COLS = ["price_usd", "peak_ccu", "dlc_count", "positive", "negative", "recommendations", "total_reviews",
                "avg_playtime_forever", "avg_playtime_two_weeks", "median_playtime_forever", "median_playtime_two_weeks"]


def detect_outliers(df: pd.DataFrame) -> None:
    print("\n[YC5] Kiểm tra outliers (không xóa)")
    stats, tops = [], []
    for c in OUTLIER_COLS:
        x = df[c].astype("float64")
        nz = x[x > 0]
        # Phân phối lệch nặng + rất nhiều số 0 -> IQR thô vô nghĩa (Q1=Q3=0). Dùng IQR trên log1p của giá trị > 0.
        lx = np.log1p(nz)
        q1, q3 = lx.quantile([.25, .75])
        iqr = q3 - q1
        mild, extreme = np.expm1(q3 + 1.5 * iqr), np.expm1(q3 + 3 * iqr)
        stats.append({"column": c, "zero_share": round(float((x == 0).mean()), 4), "n_nonzero": len(nz),
                      "median_nonzero": nz.median(), "p99": x.quantile(.99), "max": x.max(),
                      "fence_mild(1.5IQR,log)": round(mild, 2), "fence_extreme(3IQR,log)": round(extreme, 2),
                      "n_mild_or_above": int((x > mild).sum()), "n_extreme": int((x > extreme).sum())})
        top = df.nlargest(5, c)[["app_id", "name", "release_date", "owners_bucket"]].copy()
        top["value"], top["total_reviews"] = df.loc[top.index, c], df.loc[top.index, "total_reviews"]
        top.insert(0, "column", c)
        tops.append(top)

    # Quy tắc nghiệp vụ: playtime trung bình > 1 năm (525.600 phút) là đáng ngờ (treo máy / tài khoản bot)
    log_rule("YC5", "avg_playtime_forever > 1 năm (nghi treo máy)", (df["avg_playtime_forever"] > MINUTES_PER_YEAR).sum(), "GIỮ nguyên, chỉ ghi log")

    rep = pd.DataFrame(stats)
    rep.to_csv(OUT_DIR / "outliers_report.csv", index=False)
    pd.concat(tops).to_csv(OUT_DIR / "outliers_top5.csv", index=False)
    print(rep[["column", "zero_share", "p99", "max", "fence_extreme(3IQR,log)", "n_extreme"]].to_string(index=False))

In [13]:
detect_outliers(df)
top = pd.read_csv(OUT_DIR / "outliers_top5.csv")
display(top[top["column"].isin(["price_usd", "peak_ccu", "dlc_count", "avg_playtime_forever"])])


[YC5] Kiểm tra outliers (không xóa)
  [YC5] avg_playtime_forever > 1 năm (nghi treo máy): 2 -> GIỮ nguyên, chỉ ghi log
                   column  zero_share      p99        max  fence_extreme(3IQR,log)  n_extreme
                price_usd      0.2118    29.99     999.98                   262.99          7
                 peak_ccu      0.8436   215.55 1013936.00                  8191.00        109
                dlc_count      0.8567     7.00    3703.00                     9.13        810
                 positive      0.3616 12870.60 7642084.00                481760.12         32
                 negative      0.4642  2068.00 1173003.00                 94813.81         19
          recommendations      0.8330 12051.75 4830455.00                622997.39         25
            total_reviews      0.3408 15166.90 8815087.00                711779.25         26
     avg_playtime_forever      0.7920  2221.00 3429544.00                 67404.14         26
   avg_playtime_two_weeks      0.9

,column,app_id,name,release_date,owners_bucket,value,total_reviews
0,price_usd,2504210,The Leverage Game Business Edition,2023-08-26,0 - 20000,999.98,0
1,price_usd,2499620,The Leverage Game,2023-08-25,0 - 20000,999.98,1
2,price_usd,1200520,Ascent Free-Roaming VR Experience,2019-12-27,0 - 20000,999.00,7
3,price_usd,3013840,True Love,2024-06-27,0 - 20000,500.00,4
4,price_usd,1105450,Aartform Curvy 3D 4.0,2020-01-20,0 - 20000,299.99,12
5,peak_ccu,730,Counter-Strike 2,2012-08-21,100000000 - 200000000,1013936.00,8815087
6,peak_ccu,570,Dota 2,2013-07-09,100000000 - 200000000,623941.00,2498969
7,peak_ccu,578080,PUBG: BATTLEGROUNDS,2017-12-21,100000000 - 200000000,314682.00,2557944
8,peak_ccu,3419430,Bongo Cat,2025-03-05,NaN,164168.00,0
9,peak_ccu,2622380,ELDEN RING NIGHTREIGN,2025-05-29,2000000 - 5000000,163599.00,65672


## 6. Validate sau xử lý và so sánh trước-sau `[YC7]`

In [14]:
def _ws_issues(s: pd.Series) -> int:
    s = s.dropna().astype(str)
    return int((s != s.str.replace(r"\s+", " ", regex=True).str.strip()).sum())


def _dup_token_cells(s: pd.Series) -> int:
    s = s.dropna().astype(str).str.split(",")
    return int(s.map(lambda t: len(t) != len(set(t))).sum())


def metrics_before(raw: pd.DataFrame) -> dict:
    tot = raw["Positive"] + raw["Negative"]
    up = raw["Estimated owners"].str.split(" - ").str[1].astype(float)
    return {
        "rows": len(raw), "columns": raw.shape[1], "missing_cells (NaN)": int(raw.isna().sum().sum()),
        "duplicate app_id": int(raw["id"].duplicated().sum()),
        "duplicate rows (giống hệt trừ id)": int(raw.duplicated([c for c in raw.columns if c != "id"]).sum()),
        "name có khoảng trắng thừa": _ws_issues(raw["Name"]),
        "developers có token rỗng/dấu phẩy thừa": int(raw["Developers"].dropna().str.contains(r",\s*,|^,|,$").sum()),
        "categories có token lặp": _dup_token_cells(raw["Categories"]),
        "'thiếu ẩn' = 0: metacritic": int((raw["Metacritic score"] == 0).sum()),
        "'thiếu ẩn' = 0: user_score": int((raw["User score"] == 0).sum()),
        "'thiếu ẩn' = '0 - 0': owners": int((raw["Estimated owners"] == "0 - 0").sum()),
        "price=0 nhưng discount 1-99%": int(((raw["Price"] == 0) & raw["Discount"].between(1, 99)).sum()),
        "reviews/CCU > owners_max": int((((tot > up) | (raw["Peak CCU"] > up)) & (up > 0)).sum()),
        "game chưa phát hành (release_date > ngày tham chiếu)": int((pd.to_datetime(raw["Release date"], format="%b %d, %Y") > REFERENCE_DATE).sum()),
        "release_date là kiểu datetime": False,
        "memory_MB": round(raw.memory_usage(deep=True).sum() / 1e6, 1),
    }


def metrics_after(df: pd.DataFrame) -> dict:
    omax = df["owners_max"].astype("float64")
    return {
        "rows": len(df), "columns": df.shape[1], "missing_cells (NaN)": int(df.isna().sum().sum()),
        "duplicate app_id": int(df["app_id"].duplicated().sum()),
        "duplicate rows (giống hệt trừ id)": int(df.duplicated([c for c in df.columns if c != "app_id"]).sum()),
        "name có khoảng trắng thừa": _ws_issues(df["name"]),
        "developers có token rỗng/dấu phẩy thừa": int(df["developers"].str.contains(r",\s*,|^,|,$").sum()),
        "categories có token lặp": _dup_token_cells(df["categories"]),
        "'thiếu ẩn' = 0: metacritic": int((df["metacritic_score"] == 0).sum()),
        "'thiếu ẩn' = 0: user_score": int((df["user_score"] == 0).sum()),
        "'thiếu ẩn' = '0 - 0': owners": int(((df["owners_min"] == 0) & (df["owners_max"] == 0)).sum()),
        "price=0 nhưng discount 1-99%": int(((df["price_usd"] == 0) & df["discount"].between(1, 99)).sum()),
        "reviews/CCU > owners_max": int((((df["total_reviews"] > omax) | (df["peak_ccu"] > omax)) & (omax > 0)).sum()),
        "game chưa phát hành (release_date > ngày tham chiếu)": int((df["release_date"] > REFERENCE_DATE).sum()),
        "release_date là kiểu datetime": bool(pd.api.types.is_datetime64_any_dtype(df["release_date"])),
        "memory_MB": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }


def validate_and_compare(raw: pd.DataFrame, df: pd.DataFrame) -> None:
    print("\n[YC7] Validate sau xử lý")
    omax = df["owners_max"].astype("float64")
    age_min = ((REFERENCE_DATE - df["release_date"]).dt.days.clip(lower=0) + 1) * 1440
    checks = {
        "app_id duy nhất": df["app_id"].duplicated().sum(),
        "name không NA / không rỗng": df["name"].isna().sum() + (df["name"].str.strip() == "").sum(),
        "tên cột là snake_case": sum(not re.fullmatch(r"[a-z0-9_]+", c) for c in df.columns),
        "release_date là datetime & không NaT": int(df["release_date"].isna().sum()),
        "release_date >= 1990": int((df["release_date"] < "1990-01-01").sum()),
        "không còn game chưa phát hành (release_date <= ngày tham chiếu)": int((df["release_date"] > REFERENCE_DATE).sum()) if DROP_UNRELEASED else 0,
        "positive_ratio NA hoặc trong 0-1": int((df["positive_ratio"].notna() & ~df["positive_ratio"].between(0, 1)).sum()),
        "price >= 0": int((df["price_usd"] < 0).sum()),
        "discount trong 0-100": int((~df["discount"].between(0, 100)).sum()),
        "price=0 không còn discount 1-99%": int(((df["price_usd"] == 0) & df["discount"].between(1, 99)).sum()),
        "metacritic_score NA hoặc 1-100": int((df["metacritic_score"].notna() & ~df["metacritic_score"].between(1, 100)).sum()),
        "user_score NA hoặc 1-100": int((df["user_score"].notna() & ~df["user_score"].between(1, 100)).sum()),
        "required_age trong 0-21": int((~df["required_age"].between(0, 21)).sum()),
        "các cột đếm >= 0": int((df[["peak_ccu", "dlc_count", "positive", "negative", "recommendations"]] < 0).any(axis=1).sum()),
        "owners_min <= owners_max": int((df["owners_min"] > df["owners_max"]).sum()),
        "total_reviews <= owners_max (khi biết owners)": int(((df["total_reviews"] > omax) & omax.notna()).sum()),
        "peak_ccu <= owners_max (khi biết owners)": int(((df["peak_ccu"] > omax) & omax.notna()).sum()),
        "playtime <= thời gian kể từ phát hành": int((df["avg_playtime_forever"] > age_min).sum()),
        "không còn text rỗng/NA ở developers,publishers,genres,categories,tags": int(df[["developers", "publishers", "genres", "categories", "tags"]].isna().sum().sum()),
        "total_reviews = positive + negative": int((df["total_reviews"] != df["positive"] + df["negative"]).sum()),
    }
    val = pd.DataFrame([{"check": k, "violations": int(v), "status": "PASS" if v == 0 else "FAIL"} for k, v in checks.items()])
    print(val.to_string(index=False))
    val.to_csv(OUT_DIR / "validation_report.csv", index=False)

    # --- so sánh tổng quan ---
    b, a = metrics_before(raw), metrics_after(df)
    summary = pd.DataFrame({"metric": list(b), "before": list(b.values()), "after": [a[k] for k in b]})
    summary.to_csv(OUT_DIR / "before_after_summary.csv", index=False)
    print("\nSo sánh trước - sau:\n" + summary.to_string(index=False))

    # --- so sánh từng cột ---
    col_map = {c: to_snake(c) for c in raw.columns}
    col_map["Estimated owners"] = "owners_bucket"
    rows = []
    for old, new in col_map.items():
        r, d = raw[old], df[new]
        row = {"column_before": old, "column_after": new, "dtype_before": str(r.dtype), "dtype_after": str(d.dtype),
               "na_before": int(r.isna().sum()), "na_after": int(d.isna().sum())}
        if pd.api.types.is_numeric_dtype(r) and not pd.api.types.is_bool_dtype(r) and pd.api.types.is_numeric_dtype(d):
            for fn in ("mean", "median", "max"):
                row[f"{fn}_before"] = round(float(getattr(r.astype("float64"), fn)()), 3)
                row[f"{fn}_after"] = round(float(getattr(d.astype("float64"), fn)()), 3)
        rows.append(row)
    pd.DataFrame(rows).to_csv(OUT_DIR / "before_after_columns.csv", index=False)

    fails = val[val["status"] == "FAIL"]
    print("\nKẾT QUẢ VALIDATE:", "TẤT CẢ PASS" if fails.empty else f"{len(fails)} rule FAIL -> xem validation_report.csv")

In [15]:
validate_and_compare(raw, df)
display(pd.read_csv(OUT_DIR / "before_after_summary.csv"))


[YC7] Validate sau xử lý
                                                                check  violations status
                                                      app_id duy nhất           0   PASS
                                           name không NA / không rỗng           0   PASS
                                                tên cột là snake_case           0   PASS
                                 release_date là datetime & không NaT           0   PASS
                                                 release_date >= 1990           0   PASS
      không còn game chưa phát hành (release_date <= ngày tham chiếu)           0   PASS
                                     positive_ratio NA hoặc trong 0-1           0   PASS
                                                           price >= 0           0   PASS
                                                 discount trong 0-100           0   PASS
                                     price=0 không còn discount 1-99%           0   

,metric,before,after
0,rows,125855,125846
1,columns,26,33
2,missing_cells (NaN),77263,364039
3,duplicate app_id,0,0
4,duplicate rows (giống hệt trừ id),8,0
5,name có khoảng trắng thừa,304,0
6,developers có token rỗng/dấu phẩy thừa,68,0
7,categories có token lặp,96,0
8,'thiếu ẩn' = 0: metacritic,121597,0
9,'thiếu ẩn' = 0: user_score,125815,0


## 7. Lưu kết quả
`games_cleaned.csv` có 33 cột: 31 cột dữ liệu và 2 cờ cần thiết (`multi_listing_flag`, `owners_invalid`).

In [16]:
pd.DataFrame(RULE_LOG).to_csv(OUT_DIR / "cleaning_rule_log.csv", index=False)
pd.DataFrame(CHANGE_LOG).to_csv(OUT_DIR / "value_changes.csv", index=False)
df.to_csv(OUT_DIR / "games_cleaned.csv", index=False)
print(f"Đã lưu {len(df):,} dòng x {df.shape[1]} cột -> {OUT_DIR}/games_cleaned.csv")
display(df.head(3))

Đã lưu 125,846 dòng x 33 cột -> cleaning_output/games_cleaned.csv


,app_id,name,release_date,release_year,owners_bucket,owners_min,owners_max,peak_ccu,required_age,price_usd,discount,dlc_count,positive,negative,total_reviews,positive_ratio,user_score,metacritic_score,recommendations,avg_playtime_forever,avg_playtime_two_weeks,median_playtime_forever,median_playtime_two_weeks,developers,publishers,categories,genres,tags,windows,mac,linux,multi_listing_flag,owners_invalid
0,2539430,Black Dragon Mage Playtest,2023-08-01,2023,NaN,<NA>,<NA>,0,0,0.00,0,0,0,0,0,NaN,<NA>,<NA>,0,0,0,0,0,Unknown,Unknown,Unknown,Unknown,Untagged,True,False,False,False,False
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,2016,0 - 20000,0,20000,0,0,5.24,65,0,252,3,255,0.9882,<NA>,<NA>,231,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",True,False,False,False,False
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,2019,0 - 20000,0,20000,0,0,4.99,0,0,21,3,24,0.8750,<NA>,<NA>,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",True,True,False,False,False


## Quyết định đã chốt
- Xóa dòng duplicate giống hệt chỉ khác `app_id` (8 dòng), giữ `app_id` nhỏ nhất.
- Xóa game chưa phát hành theo `REFERENCE_DATE` (1 dòng). Đổi ngày tham chiếu ở ô cấu hình đầu notebook.
- Giữ `user_score` (chỉ 40 giá trị thật); dùng `positive_ratio` để phân tích điểm đánh giá. Khi phân tích nên lọc `total_reviews` từ khoảng 10 trở lên.
- Đơn giá là USD (`price_usd`).
- Outliers giữ nguyên. Các giá trị cực trị phần lớn là thật (Counter-Strike 2, Dota 2, Fantasy Grounds...).
- Owners chỉ là khoảng ước lượng (ví dụ 20.000-50.000), không phải số chính xác.